# 📊 Evaluation Metrics for Chart/Document VQA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/production-vlm-engineering/blob/main/notebooks/colab/01_evaluation_metrics_colab.ipynb)

Part of **[production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)** — a production-grade VLM engineering playbook.

This notebook demonstrates why standard text metrics (BLEU, exact-match) fail for chart/document VQA, and how `numeric_accuracy`, `grounding_score`, and `faithfulness_score` fix this.

**No GPU required. Runtime: ~2 minutes.**

In [1]:
# ── Setup ──────────────────────────────────────────────────────────
# Install the production-vlm-engineering package (CPU-only core)
import subprocess
result = subprocess.run(
    ["pip", "install", "git+https://github.com/Mattral/production-vlm-engineering.git",
     "--quiet", "--no-deps"],
    capture_output=True, text=True
)
# Fallback: install direct dependencies if package install fails
if result.returncode != 0:
    subprocess.run(["pip", "install", "numpy", "scipy", "matplotlib", "Pillow", "pyyaml",
                    "--quiet"], check=True)
    import subprocess as _sp
    _sp.run(["git", "clone", "--depth=1", "https://github.com/Mattral/production-vlm-engineering.git",
             "/content/production_vlm_repo"], capture_output=True)
    import sys
    sys.path.insert(0, "/content/production_vlm_repo/src")
    sys.path.insert(0, "/content/production_vlm_repo")

print("✅ Setup complete")

✅ Setup complete


In [2]:
from production_vlm.eval import numeric_accuracy, grounding_score, faithfulness_score
print("Imports OK")

Imports OK


## Why BLEU and exact-match fail on chart answers

Consider a model that correctly reads a chart showing 4.2M revenue.
The prediction and reference are semantically identical but formatted differently.

In [3]:
prediction = "The chart shows revenue of $4.21M"
reference  = "Revenue was 4.2 million dollars"

result = numeric_accuracy(prediction, reference, tolerance=0.02)
print(f"numeric_accuracy score: {result.score:.1f}  ✅ (BLEU would score ≈ 0.0)")
print(f"Matched: {result.matched}/{result.total_reference} numbers within ±{result.tolerance:.0%}")

numeric_accuracy score: 1.0  ✅ (BLEU would score ≈ 0.0)
Matched: 1/1 numbers within ±2%


## Partial credit and tolerance

In [4]:
result = numeric_accuracy("North: 50 units, South: 67.6, West: 999",
                         "North: 50.0 units, South: 67.6 units, East: 30.2 units")
print(f"Partial match: {result.matched}/{result.total_reference} → score={result.score:.3f}")

Partial match: 2/3 → score=0.667


## Grounding score — is the answer actually referencing the chart?

In [5]:
evidence = "South: 67.6 req/s; North: 50.0 req/s; East: 45.2 req/s"

r_good = grounding_score("South region showed throughput of 67.6 req/s", evidence)
r_hall = grounding_score("The metrics indicate a moderate baseline across all regions", evidence)

print(f"Grounded answer:     {r_good.score:.2f}  ({r_good.grounded_terms}/{r_good.total_terms} terms)")
print(f"Hallucinated answer: {r_hall.score:.2f}  ({r_hall.grounded_terms}/{r_hall.total_terms} terms)")

Grounded answer:     0.71  (5/7 terms)
Hallucinated answer: 0.00  (0/5 terms)


## Composite faithfulness score

In [6]:
from production_vlm.utils.synthetic_charts import generate_synthetic_chart

chart = generate_synthetic_chart(seed=5, render_image=False)
print(f"Chart: {chart.title}")
print(f"Evidence: {chart.evidence_text}")
print()

cases = [
    ("Correct (paraphrased)", chart.answer.replace("has","shows")),
    ("Vague / ungrounded",    "The values appear moderate based on the chart data."),
    ("Wrong numbers",         f"{chart.categories[0]} has a value of {chart.values[-1]*3:.1f} {chart.units}, the highest."),
]

print(f"{'Prediction':<28} {'Faith':>6} {'Num':>6} {'Ground':>8}")
print("-"*52)
for label, pred in cases:
    r = faithfulness_score(pred, chart.answer, chart.evidence_text)
    print(f"{label:<28} {r.score:>6.3f} {r.numeric.score:>6.3f} {r.grounding.score:>8.3f}")

Chart: Conversion Rate by Quarter
Evidence: Q2: 97.1 %; Q3: 97.8 %; Q1: 89.3 %; Q4: 74.4 %

Prediction                   Faith    Num   Ground
----------------------------------------------------
Correct (paraphrased)        0.720  0.800    0.600
Vague / ungrounded           0.000  0.000    0.000
Wrong numbers                0.200  0.000    0.500


## Try it yourself! 🎯

Change the prediction text below and see how the metrics respond:

In [7]:
YOUR_PREDICTION = "Q3 shows the highest conversion rate of 97.8%"
YOUR_REFERENCE  = chart.answer
YOUR_EVIDENCE   = chart.evidence_text

result = faithfulness_score(YOUR_PREDICTION, YOUR_REFERENCE, YOUR_EVIDENCE)
print(f"Faithfulness: {result.score:.3f}")
print(f"  Numeric accuracy: {result.numeric.score:.3f}")
print(f"  Grounding score:  {result.grounding.score:.3f}")

---
**Next**: [02 — Drift Detection & Active Learning](02_drift_detection_colab.ipynb)

**Repo**: [github.com/Mattral/production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)  
**Reference**: Es et al. (2023) *RAGAS* · arXiv:2309.15217